In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/titanic/train.csv
/kaggle/input/competitions/titanic/test.csv
/kaggle/input/competitions/titanic/gender_submission.csv


# 最終モデル
notebookの09_model-tuningより
* 使用モデル:XGBClassifier
* 使う特徴量:
  >数値列 =["logFare","Family","has_cabin","Pclass"]
  >
>カテゴリ列 = ["Embarked","Sex","age_binning"]

* 

In [2]:
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import cross_val_score

# データ読み込み

In [3]:
#"PassengerId"をインデックスに指定
train_csv = pd.read_csv("/kaggle/input/competitions/titanic/train.csv").set_index("PassengerId")
test_csv = pd.read_csv("/kaggle/input/competitions/titanic/test.csv").set_index("PassengerId")

y = train_csv.Survived
X = train_csv.drop("Survived", axis=1)

# 特徴量の作成
* 新しく作った特徴量
  >"Title":"Name"より、敬称のみ抽出
  >
  >"Age":欠損を"Title"別の中央値で補填
  >
  >"Family":SibSp,Parchはそれぞれ親戚・家族のことを表し、行動を一にしていた可能性が高いためまとめたものを作成
  >
  >"logFare":運賃を対数変換し見やすく
  >
  >"has_cabin":船室情報の有無自体が船室ランク、ひいては生存率への情報へとつながるため作成。
  >
  >"age_binning":年齢帯による生存率の変化が大きいことから、4つに分類分け

In [4]:
class TitanicFeatureEngineering(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass

    def fit(self,X,y=None):
        X = X.copy()

        X["Title"] = X["Name"].str.extract(r" ([A-Za-z]+)\.", expand=False)
        X["Title"] = X["Title"].replace(["Mlle","Ms"], "Miss")
        X["Title"] = X["Title"].replace("Mme", "Mrs")
        rare_titles = [
             "Lady", "Countess", "Capt", "Col", "Don", "Dr", "Major",
            "Rev", "Sir", "Jonkheer", "Dona"
        ]
        X["Title"] = X["Title"].replace(rare_titles, "Rare")

        self.title_age_median = X.groupby("Title")["Age"].median()
        self.global_age_median = X["Age"].median()

        return self

    def transform(self,X):
        X_new = X.copy()

        X_new["Family"] = X_new["Parch"].astype("Int64") + X_new["SibSp"].astype("Int64")

        X_new["logFare"] = np.log1p(X_new["Fare"])#1人分の運賃で計算

        X_new["has_cabin"] = X_new["Cabin"].notnull().astype(int)

        X_new["Title"] = X_new["Name"].str.extract(r" ([A-Za-z]+)\.",expand = False)
        X_new["Title"] = X_new["Title"].replace(["Mlle","Ms"],"Miss")
        X_new["Title"] = X_new["Title"].replace("Mme","Mrs")
        rare_titles = [
            "Lady", "Countess", "Capt", "Col", "Don", "Dr", "Major",
            "Rev", "Sir", "Jonkheer", "Dona"
        ]
        X_new["Title"] = X_new["Title"].replace(rare_titles,"Rare")

        X_new["Age"] = X_new["Age"].fillna(X_new["Title"].map(self.title_age_median))
        X_new["Age"] = X_new["Age"].fillna(self.global_age_median) #未知の値の対処

        child = X_new["Age"]<12
        young = (X_new["Age"]>=12) & (X_new["Age"]<20)
        adult = (X_new["Age"]>=20) & (X_new["Age"]<60)
        senior = X_new["Age"]>=60

        age_selection = [child,young,adult,senior]
        name = ["child","young","adult","senior"]
        X_new["age_binning"] = np.select(age_selection,name,default ="unknown") #未知の値の対処

        return X_new
        

# Pipelineの作成

In [5]:
numerical_transformer = Pipeline(
    steps = [
        ("imputer", SimpleImputer(strategy="median")) #中央値埋め
    ]
)

categorical_transformer = Pipeline(
    steps = [
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output = False)) #未知の値の対処、密行列で返す
    ]
)

num_cols =["logFare","Family","has_cabin","Pclass"] #数値列
cat_cols = ["Embarked","Sex","age_binning"] #カテゴリ列

preprocessor = ColumnTransformer(
    transformers = [
        ("num", numerical_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols)
    ]
)

# 最適パラメータの探索にOptuna使用
cvは分割時の詳細設定のため

In [6]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(
    n_splits=5, #5分割
    shuffle=True, #分散させるため
    random_state=10 #同じ分散にするため
)

# XGBClassifierモデルの設定
パラメータは、optunaを使用し、一番スコアが良かったものを採用。

In [7]:
from xgboost import XGBClassifier

XGBC_pipeline = Pipeline(
    steps = [
        ("feature_engineering", TitanicFeatureEngineering()),
        ("preprocessor",preprocessor),
        ("model",XGBClassifier(
            n_jobs = -1, #最大コア使用
            random_state = 30, #設定維持のため
            eval_metric = "logloss", #警告文を消すため
            learning_rate = 0.03, #学習率
            n_estimators = 300 #決定木の数
        ))
])

In [8]:
def XGBC_func(trial):
    params = {
        "colsample_bytree":trial.suggest_float("colsample_bytree",0.6,1.0),
        "reg_alpha":trial.suggest_float("reg_alpha",1e-2,2.0,log=True),
        "max_depth":trial.suggest_int("max_depth",3,6),
        "min_child_weight":trial.suggest_int("min_child_weight",2,5),
        "subsample":trial.suggest_float("subsample",0.6,1.0)
    }

    #Pipelineへのパラメータ探索のセット
    XGBC_pipeline.set_params(**{f"model__{k}":v for k,v in params.items()} )

    scores = cross_val_score(XGBC_pipeline,X,y,cv=cv,scoring="accuracy")

    return scores.mean()

study = optuna.create_study(direction = "maximize")
study.optimize(XGBC_func,n_trials=50,show_progress_bar=True)

print(f"ベストスコア:{study.best_value}")
print(f"ベストパラメータ:{study.best_params}")

XGBC_best_value=study.best_value
XGBC_best_params = study.best_params

  0%|          | 0/50 [00:00<?, ?it/s]

ベストスコア:0.8439708743958321
ベストパラメータ:{'colsample_bytree': 0.714633662242447, 'reg_alpha': 0.12285991494537762, 'max_depth': 6, 'min_child_weight': 2, 'subsample': 0.8580209262254561}


In [9]:
XGBC_pipeline.fit(X,y)
preds = XGBC_pipeline.predict(test_csv)

submission = pd.DataFrame({"PassengerId":test_csv.index, "Survived":preds})
submission.to_csv("submission.csv", index=False)